# 01 · Ympäristö ja testit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markomanninen/tgd-geometry-lab/blob/main/notebooks/01_ymparisto_ja_testit.ipynb)

Tämä kirja on rakennettu Lanyonin Colab-esimerkin (*Run LanyonScripts in Colab*) mallin mukaan:
väliaikainen Ubuntu-ajoympäristö, GNU Maxima, Prodanovin `clifford`/`cliffordan` -paketit ja
skriptien ajo omissa Maxima-prosesseissaan.

Erona Lanyonin esimerkkiin: jokainen moduuli tarkistetaan **automaattisilla testeillä**
(tunnetut identiteetit kuten $d^2=0$, $d\omega=0$, $\mathrm{Ric}=3g$, Bianchi), jotta
laskentakirjaston virheet eivät sekoitu TGD:n fysikaalisiin oletuksiin (välitavoite M1).

In [1]:
# Asennus: Maxima, tämä repositorio ja Dimiter Prodanovin clifford-paketti.
# Colabissa ajo kestää noin minuutin; paikallisesti asetetaan TGD_ROOT / CLIFFORD_PATH.
import os, sys, shutil
ROOT = os.environ.get("TGD_ROOT", "/content/tgd-geometry-lab")
CLI  = os.environ.get("CLIFFORD_PATH", "/content/clifford")
if not shutil.which("maxima"):
    !apt-get -qq update > /dev/null && apt-get -qq install -y maxima > /dev/null
if not os.path.exists(ROOT):
    !git clone -q --depth 1 https://github.com/markomanninen/tgd-geometry-lab.git {ROOT}
if not os.path.exists(CLI):
    !git clone -q --depth 1 https://github.com/dprodanov/clifford.git {CLI}
sys.path.insert(0, ROOT + "/notebooks")
from tgdlab import Lab
lab = Lab(ROOT, CLI)
!maxima --version 2>/dev/null | tail -1

Maxima 5.50.0


## Testisarja

Ajaa kaikki `tests/`-hakemiston testit (metriikka, kaarevuus, muodot, $CP_2$, upotus, toiminta, Clifford).

In [2]:
ok = lab.tests()
print('KAIKKI TESTIT LÄPI' if ok else 'TESTEJÄ EPÄONNISTUI')

or M^4) 
  PASS  cosmic string: induced metric = M^2 (+) (-) S^2(r=1/2) 
  PASS  cosmic string: J_ab J^ab = 2 (constant magnetic Kahler field) 
  PASS  cosmic string: Kahler magnetic flux density J_ab = 1/(1+r^2)^2 da^db 
--- embedding/second-form: B, H^k, Gauss, Codazzi, Ricci --- 
  PASS  sphere: g = 4 rho^2 delta / (1+u^2+v^2)^2 
  PASS  sphere: B^k_ab = -g_ab n^k / rho 
  PASS  sphere: H^k = -2 n^k / rho 
  PASS  sphere: B is normal to the surface 
  PASS  cylinder: H^k = -n^k / rho (one principal curvature) 
  PASS  cylinder in polar ambient coordinates: H^r = -1/rho 
  PASS  hyperboloid in R^(1,2): H^k = -2 h^k / rho^2 * (n.n) 
  PASS  helicoid (ruled minimal surface): H^k = 0 
  PASS  CP_1 = {z2 = 0} is totally geodesic: B = 0 
  PASS  conic z2 = z1^2 is not totally geodesic: B # 0 
  PASS  conic z2 = z1^2 (complex curve) is minimal: H^k = 0 
  PASS  non-holomorphic graph z2 = a^2 is not minimal 
  PASS  Gram-Schmidt on the sphere: one unit normal, +-n 
  PASS  Gram-Schmidt on C

## Clifford-algebra (Prodanov)

Sama paketti kuin LanyonScriptsissä. Tarkistetaan Clifford-relaatiot
$\gamma_a\gamma_b+\gamma_b\gamma_a = 2\eta_{ab}$ signatuurilla $(1,3)$.

In [3]:
print(lab.values("clifford(g, 1, 3)$",
    ["g[1] . g[1]", "g[2] . g[2]", "g[1] . g[2] + g[2] . g[1]", "g[3] . g[4] + g[4] . g[3]"],
    load_clifford=True))

['1', '-1', '0', '0']


## Vertailu: LanyonScriptsin esimerkit

Valinnainen: ajetaan Lanyonin kuusi virtausfysiikan esimerkkiä (Burgers, matala vesi, Euler, Maxwell, MHD)
samalla tavalla kuin alkuperäisessä kirjassa. Ne käyttävät samaa Maxima–Clifford-pohjaa,
jonka päälle TGD Geometry Lab rakentaa differentiaaligeometrian.

In [4]:
RUN_LANYON = os.environ.get("RUN_LANYON", "1") == "1"   # aseta 0 ohittaaksesi
if RUN_LANYON:
    import subprocess
    from pathlib import Path
    L = Path("/content/LanyonScripts")
    if not L.exists():
        !git clone -q --depth 1 https://github.com/lanyonai/LanyonScripts.git {L}
    for f in ["burgers.mac", "shallow-water.mac", "isoeuler.mac", "euler.mac", "maxwell.mac", "ideal-mhd.mac"]:
        code = (f'file_search_maxima: append(file_search_maxima, ["{CLI}/###.{{mac,wxm}}", "{CLI}/$$$.{{mac,wxm}}"])$ '
                f'load("{L}/eqnsys/{f}")$ print("LANYON_SCRIPT_FINISHED")$')
        p = subprocess.run(["maxima", "--very-quiet", f"--batch-string={code}"],
                           capture_output=True, text=True, timeout=300)
        print(f"{f:20s}", "OK" if "LANYON_SCRIPT_FINISHED" in p.stdout else "VIRHE")